# AWS Glue Job Bookmark

## The Problem Without Job Bookmark

You have orders files landing in S3 every day:
```
s3://my-bucket/raw/orders/orders_day1.csv   <- arrived Day 1
s3://my-bucket/raw/orders/orders_day2.csv   <- arrived Day 2
s3://my-bucket/raw/orders/orders_day3.csv   <- arrived Day 3
```

Your Glue job runs every day to process these files.

**Without Job Bookmark:**
```
Run 1 (Day 1): processes orders_day1.csv                          → target has 6 records
Run 2 (Day 2): processes orders_day1.csv + orders_day2.csv AGAIN  → target has 18 records (duplicates!)
Run 3 (Day 3): processes all 3 files AGAIN                        → target has 36 records (more duplicates!)
```

**With Job Bookmark:**
```
Run 1 (Day 1): processes orders_day1.csv              → bookmark saves: day1 done
Run 2 (Day 2): skips orders_day1.csv, reads day2 only → bookmark saves: day1, day2 done
Run 3 (Day 3): skips day1 and day2, reads day3 only   → bookmark saves: day1, day2, day3 done
```

---
## Our Data
Each file has this structure:
```
order_id,order_date,order_customer_id,order_status
1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
...
```
We simulate 3 daily drops:
- `orders_day1.csv` — 6 records
- `orders_day2.csv` — 6 records (uploaded after Run 1)
- `orders_day3.csv` — 6 records (uploaded after Run 2)

---
## Setup

In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# In a real Glue job this comes from: getResolvedOptions(sys.argv, ['JOB_NAME'])
JOB_NAME = "orders_incremental_load"

INPUT_PATH  = "s3://my-bucket/raw/orders/"
OUTPUT_PATH = "s3://my-bucket/silver/orders/"

print("Setup complete")

---
## The Two Lines That Make Job Bookmark Work

```python
job.init(JOB_NAME, args)   # tells Glue: "load the last saved bookmark state for this job"
job.commit()               # tells Glue: "save the new bookmark state after this run"
```

If you forget `job.commit()` → bookmark is never saved → next run reprocesses everything again.

**How Glue tracks files internally:**
```
After Run 1, Glue stores this internally:
{
  "orders_day1.csv": { "etag": "abc123", "size": 245 }
}

On Run 2, Glue compares S3 file list against this bookmark:
- orders_day1.csv → already in bookmark → SKIP
- orders_day2.csv → NOT in bookmark    → PROCESS
```

---
## RUN 1 — Only orders_day1.csv exists in S3

In [ ]:
# S3 state at this point:
# s3://my-bucket/raw/orders/orders_day1.csv  (6 records)

job = Job(glueContext)
job.init(JOB_NAME, {})

datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [INPUT_PATH], "recurse": True},
    format="csv",
    format_options={"withHeader": True}
)

print(f"Records read in Run 1: {datasource.count()}")
datasource.show()

glueContext.write_dynamic_frame.from_options(
    frame=datasource,
    connection_type="s3",
    connection_options={"path": OUTPUT_PATH},
    format="parquet"
)

job.commit()  # bookmark now remembers: orders_day1.csv is processed

**Expected Output:**
```
Records read in Run 1: 6

{"order_id": "1", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11599", "order_status": "CLOSED"}
{"order_id": "2", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "256",   "order_status": "PENDING_PAYMENT"}
{"order_id": "3", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "12111", "order_status": "COMPLETE"}
{"order_id": "4", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "8827",  "order_status": "CLOSED"}
{"order_id": "5", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11318", "order_status": "COMPLETE"}
{"order_id": "6", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "7130",  "order_status": "COMPLETE"}
```
> Bookmark state saved: `orders_day1.csv` is marked as processed.
> Target now has **6 records**.

---
## RUN 2 — orders_day2.csv is now uploaded to S3

S3 state now:
```
s3://my-bucket/raw/orders/orders_day1.csv  <- already processed (in bookmark)
s3://my-bucket/raw/orders/orders_day2.csv  <- NEW file
```

In [ ]:
job = Job(glueContext)
job.init(JOB_NAME, {})

# Glue reads INPUT_PATH which has BOTH day1 and day2
# But bookmark tells it to skip day1 automatically
datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [INPUT_PATH], "recurse": True},
    format="csv",
    format_options={"withHeader": True}
)

print(f"Records read in Run 2: {datasource.count()}")
datasource.show()

glueContext.write_dynamic_frame.from_options(
    frame=datasource,
    connection_type="s3",
    connection_options={"path": OUTPUT_PATH},
    format="parquet"
)

job.commit()  # bookmark now remembers: orders_day1.csv + orders_day2.csv processed

**Expected Output:**
```
Records read in Run 2: 6    <-- only day2 records, NOT 12. day1 was skipped by bookmark.

{"order_id": "7",  "order_date": "2013-07-26 00:00:00.0", "order_customer_id": "5000", "order_status": "COMPLETE"}
{"order_id": "8",  "order_date": "2013-07-26 00:00:00.0", "order_customer_id": "5001", "order_status": "CLOSED"}
...
```
> `orders_day1.csv` was in the bookmark → Glue skipped it entirely.
> Only `orders_day2.csv` was read and written.
> Target now has **12 records total** (6 from Run 1 + 6 from Run 2). No duplicates.

---
## RUN 3 — No new files uploaded

S3 state now:
```
s3://my-bucket/raw/orders/orders_day1.csv  <- already processed
s3://my-bucket/raw/orders/orders_day2.csv  <- already processed
```
No new file arrived today. What happens?

In [ ]:
job = Job(glueContext)
job.init(JOB_NAME, {})

datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [INPUT_PATH], "recurse": True},
    format="csv",
    format_options={"withHeader": True}
)

record_count = datasource.count()
print(f"Records read in Run 3: {record_count}")

if record_count == 0:
    print("No new files to process. Skipping write.")
else:
    glueContext.write_dynamic_frame.from_options(
        frame=datasource,
        connection_type="s3",
        connection_options={"path": OUTPUT_PATH},
        format="parquet"
    )

job.commit()

**Expected Output:**
```
Records read in Run 3: 0
No new files to process. Skipping write.
```
> Both files are in the bookmark. Glue returns an empty DynamicFrame.
> Target still has **12 records**. Nothing changed.

---
## What Happens If You Forget job.commit()?

In [ ]:
# BAD EXAMPLE — do not do this
job = Job(glueContext)
job.init(JOB_NAME, {})

datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [INPUT_PATH], "recurse": True},
    format="csv",
    format_options={"withHeader": True}
)

print(f"Records read: {datasource.count()}")

# job.commit() is missing here
# Bookmark state is NOT saved
# Next run will read the same files again

**Expected Output:**
```
Records read: 6   <-- reads day1 again because bookmark was never saved from last run
```
> Without `job.commit()`, Glue does not save the bookmark.
> Every run will reprocess the same files → duplicates in target.

---
## Resetting the Bookmark

Use this when:
- You had a bug and need to reprocess all historical files
- Target data was accidentally deleted
- Schema changed and all data needs to be rewritten

In [ ]:
import boto3

glue_client = boto3.client('glue', region_name='ap-south-1')
glue_client.reset_job_bookmark(JobName=JOB_NAME)

print(f"Bookmark reset for job: {JOB_NAME}")
print("Next run will reprocess ALL files from the beginning")

**Expected Output:**
```
Bookmark reset for job: orders_incremental_load
Next run will reprocess ALL files from the beginning
```
> After reset, the next run will read `orders_day1.csv + orders_day2.csv` again.
> Make sure to clear the target first, otherwise you will get duplicates.

---
## Viewing the Current Bookmark State

In [ ]:
import boto3, json

glue_client = boto3.client('glue', region_name='ap-south-1')
response = glue_client.get_job_bookmark(JobName=JOB_NAME)

print(json.dumps(response['JobBookmarkEntry'], indent=2))

**Expected Output (after Run 2):**
```json
{
  "JobName": "orders_incremental_load",
  "Version": 2,
  "Run": 2,
  "JobBookmark": "{\"orders_day1.csv\": {\"etag\": \"abc123\", \"size\": 245}, \"orders_day2.csv\": {\"etag\": \"def456\", \"size\": 251}}"
}
```
> The `JobBookmark` field lists every file Glue has already processed with its ETag and size.
> On the next run, Glue compares the S3 file list against this and skips anything already here.

---
## Summary

| | Without Bookmark | With Bookmark |
|---|---|---|
| Run 1 (day1 only) | reads day1 → 6 records | reads day1 → 6 records |
| Run 2 (day1 + day2) | reads day1 + day2 → 12 records (**duplicates**) | reads day2 only → 6 records |
| Run 3 (no new file) | reads day1 + day2 → 12 records (**more duplicates**) | reads nothing → 0 records |
| Target after 3 runs | **30 records** (duplicates everywhere) | **12 records** (clean) |

## Rules to Remember
1. `job.init()` loads the last saved bookmark state
2. `job.commit()` saves the new bookmark state — **never skip this**
3. Always use `mode("append")` on the target when using bookmarks
4. Job Bookmark only works with `create_dynamic_frame` — NOT with `spark.read`
5. Reset bookmark + clear target together, never one without the other